# Example 1.1.1 / 4: Indexed Set Methods in amplpy

Companion notebook for **1.1.1 Flat vs Indexed Sets** in the AMPL lab Documentation, section "Indexed Set Methods in amplpy". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Indexed Set Methods in amplpy

The `Set` object of an indexed set, `ampl.set["AREA"]`, gives access to its members, one `Set`
object for each index:

| Method | What it does | Pure AMPL |
|---|---|---|
| `ampl.set["S"] = {i: [...], ...}` | gives every `S[i]` its members, from a dictionary of lists | `set S[i] := ...;` in `data.dat` |
| `ampl.set["S"][i]` | the one set `S[i]`, with `to_list()`, `size()` and the other `Set` methods | `S[i]` |
| `instances()` | every pair (index, `Set` object) of the family | `display S;` |
| `num_instances()` | how many sets the family holds | `card` of the indexing set |

`instances()` returns each index as a tuple, `('bands',)`, because an indexed set may have more
than one subscript:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set AREA {PROD};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["AREA"] = {"bands": ["east", "north"], "coils": ["east", "west", "export"]}

In [5]:
# run.py
for index, areas in ampl.set["AREA"].instances():
    print(index, areas.to_list(), areas.size())
print(ampl.set["AREA"].num_instances())

('bands',) ['east', 'north'] 2
('coils',) ['east', 'west', 'export'] 3
2


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set AREA[bands] := east north;
set AREA[coils] := east west export;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display AREA;
display {p in PROD} card(AREA[p]);
display card(PROD);

set AREA[bands] := east north;
set AREA[coils] := east west export;

card( AREA[p]) [*] :=
bands  2
coils  3
;

card(PROD) = 2

